# NB04 — Agentic RAG Pipeline (Method 2)
**MSK | Goel Lab** · LLM Systematic Review · Method Comparison

**Purpose**: Run the LangGraph-based agentic RAG agent. The agent decomposes the review question into sub-queries, iteratively retrieves and re-ranks, then synthesises. Run 3× to assess reproducibility (Jaccard similarity).

**Stack**: LangGraph + LangChain 0.3.x + GPT-4o (T=0)

**Outputs**
- `data/processed/agentic_rag_results.csv`
- `data_private/method_outputs/agentic_rag/` — full synthesis texts
- `experiments/<run_id>.json` (×3 runs)

In [ ]:
import sys
sys.path.insert(0, '..')

from src.utils.io import load_env, data_dir, generate_run_id, log_experiment
load_env()

from pathlib import Path
import pandas as pd
import time

## 1. Initialise agent (reuses vector store from NB01)

In [ ]:
from src.methods.standard_rag.langchain_pipeline import load_vector_store, build_rag_chain
from src.methods.standard_rag.pipeline import StandardRAGPipeline
from src.methods.agentic_rag.agent import AgenticRAGAgent
import numpy as np

vs = load_vector_store()
_, retriever = build_rag_chain(vs)

# Wrap LangChain retriever for the agent's vector_pipeline interface
class LangChainRetrieverWrapper:
    def retrieve(self, query, top_k=10):
        docs = retriever.invoke(query)[:top_k]
        return pd.DataFrame([{
            'doc_id': d.metadata.get('source', ''),
            'text':   d.page_content,
        } for d in docs])

agent = AgenticRAGAgent(vector_pipeline=LangChainRetrieverWrapper())
print('Agent ready.')

## 2. Run 3× for reproducibility assessment

In [ ]:
REVIEW_QUESTION = (
    'What LLM optimization strategies (prompt engineering, fine-tuning, RAG, '
    'chain-of-thought, instruction tuning) have been proposed or evaluated for '
    'extracting structured clinical features from radiology and pathology reports?'
)

run_results = []
for i in range(3):
    t0 = time.perf_counter()
    run_id = generate_run_id('agentic_rag')
    result = agent.run(question=REVIEW_QUESTION, run_id=run_id)
    result['time_seconds'] = round(time.perf_counter() - t0, 1)
    run_results.append(result)
    log_experiment(result)
    print(f'Run {i+1}: {result["n_retrieved"]} docs, {result["n_iterations"]} iterations, {result["time_seconds"]}s')

## 3. Jaccard reproducibility across 3 runs

In [ ]:
sets = [set(r['retrieved_doc_ids']) for r in run_results]
jaccards = []
for i in range(len(sets)):
    for j in range(i+1, len(sets)):
        inter = len(sets[i] & sets[j])
        union = len(sets[i] | sets[j])
        j_sim = inter / union if union else 0
        jaccards.append(j_sim)
        print(f'Run {i+1} vs Run {j+1}: Jaccard = {j_sim:.3f}')
print(f'Mean Jaccard: {sum(jaccards)/len(jaccards):.3f}')

## 4. Save results

In [ ]:
summary_rows = [{
    'run_id':        r['run_id'],
    'n_retrieved':   r['n_retrieved'],
    'n_iterations':  r['n_iterations'],
    'time_seconds':  r['time_seconds'],
} for r in run_results]

pd.DataFrame(summary_rows).to_csv(
    data_dir() / 'processed' / 'agentic_rag_results.csv', index=False
)
print('Saved.')